[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_9/02_tag_9_semantische_segmentierung_resnet_unet.ipynb)

# Tag 9 – 02: Semantische Segmentierung mit ResNet-18 und U-Net-Decoder

Bei der **Objekterkennung** aus Notebook 01 ist das Label eine Bounding Box. Hier ist das Label eine Maske: Für **jedes Pixel** entscheidet das Modell zwischen Hintergrund (`0`) und Person (`1`). Wir verwenden dieselben echten Penn-Fudan-Fotos und Masken aus Notebook 00.

Das Modell kombiniert einen ImageNet-vortrainierten **ResNet-18-Backbone** (Encoder) mit einem neu initialisierten, U-Net-artigen **Decoder**. Zuerst trainieren wir nur Decoder und Segmentierungs-Head. Danach wird der letzte ResNet-Block vorsichtig mittrainiert.

## Lernziel: Bild, Maske und Vorhersage

| Rolle | Form bei `IMAGE_SIZE = 224` | Bedeutung |
| --- | --- | --- |
| Input `image` | `(3, 224, 224)` | RGB-Foto, nur die drei Farbkanäle |
| Label `mask` | `(224, 224)` | pro Pixel `0` (Hintergrund) oder `1` (Person) |
| Modell-Logits | `(2, 224, 224)` | zwei unnormierte Werte pro Pixel |
| Vorhersage | `(224, 224)` | Klasse mit der höchsten Wahrscheinlichkeit pro Pixel |

Die Originalmasken kodieren verschiedene Personen mit verschiedenen positiven IDs. Für **semantische** Segmentierung interessieren uns hier nur zwei Klassen; daher werden alle positiven IDs zu `1` zusammengefasst. Damit kann das Modell auch Bilder mit mehreren Personen nutzen.

In [ ]:
from pathlib import Path
import json
import random

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18
from torchvision.transforms import InterpolationMode, functional as TF

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

DATASET_ROOT = Path('datasets/PennFudanPed')
IMAGE_SIZE = 224
BATCH_SIZE = 4
HEAD_EPOCHS = 6
FINE_TUNE_EPOCHS = 4
USE_PRETRAINED_BACKBONE = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('PyTorch:', torch.__version__)
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Datensatz aus Notebook 00 laden

Führe vorher `00_tag_9_datensatz_vorbereiten.ipynb` aus. Dieses Notebook erwartet dessen `segmentation_train.json` und `segmentation_valid.json`. Die Fotos werden für das Modell auf 224×224 skaliert. Die Maske wird dabei mit **Nearest-Neighbour** skaliert: So bleiben ihre Klassenwerte `0` und `1` erhalten; bilineares Interpolieren würde ungültige Zwischenwerte erzeugen.

Im Training spiegeln wir Bild und Maske gemeinsam horizontal. Helligkeit und Kontrast ändern wir ausschließlich im Bild – das Label bleibt geometrisch identisch.

In [ ]:
class PennFudanSegmentationDataset(Dataset):
    def __init__(self, split, augment=False):
        annotation_path = DATASET_ROOT / f'segmentation_{split}.json'
        if not annotation_path.exists():
            raise FileNotFoundError(
                f'{annotation_path} fehlt. Bitte zuerst Notebook 00 ausführen.'
            )
        self.samples = json.loads(annotation_path.read_text(encoding='utf-8'))
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        image = Image.open(DATASET_ROOT / sample['image']).convert('RGB')
        instance_mask = Image.open(DATASET_ROOT / sample['mask'])

        image = TF.resize(TF.to_tensor(image), [IMAGE_SIZE, IMAGE_SIZE], antialias=True)
        # Alle Instanz-IDs > 0 werden zur Person-Klasse 1.
        mask = torch.from_numpy((np.asarray(instance_mask) > 0).astype(np.int64))
        mask = TF.resize(mask.unsqueeze(0), [IMAGE_SIZE, IMAGE_SIZE],
                         interpolation=InterpolationMode.NEAREST).squeeze(0).long()

        if self.augment:
            if random.random() < 0.5:
                image = TF.hflip(image)
                mask = TF.hflip(mask)
            image = TF.adjust_brightness(image, random.uniform(0.85, 1.15))
            image = TF.adjust_contrast(image, random.uniform(0.85, 1.15))
        return image, mask

train_dataset = PennFudanSegmentationDataset('train', augment=True)
valid_dataset = PennFudanSegmentationDataset('valid')
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

image, mask = valid_dataset[0]
print(f'Training: {len(train_dataset)} Bilder; Validierung: {len(valid_dataset)} Bilder')
print('Input:', tuple(image.shape), image.dtype)
print('Maskenklassen:', torch.unique(mask).tolist())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(image.permute(1, 2, 0))
axes[0].set_title('Input: echtes Foto')
axes[1].imshow(mask, cmap='magma', vmin=0, vmax=1)
axes[1].set_title('Label: 0 Hintergrund, 1 Person')
for ax in axes:
    ax.axis('off')
plt.tight_layout()

## Encoder + Decoder: vortrainierter Backbone, neuer Segmentierungs-Teil

Ein ResNet-18 lernt auf ImageNet bereits nützliche Kanten, Texturen und Formen. Sein Klassifikations-Head wäre für eine einzelne Bildklasse gedacht und wird entfernt. Stattdessen behalten wir seine räumlichen Zwischenrepräsentationen (`layer1` bis `layer4`).

Der neue Decoder vergrößert die kleinen Feature Maps schrittweise. An jeder Stufe verbindet er sie mit einer höher aufgelösten ResNet-Repräsentation (**Skip Connection**). Das entspricht der zentralen U-Net-Idee: tiefe Features geben den Kontext, frühe Features helfen bei präzisen Kanten.

| Phase | Trainierbar | Zweck |
| --- | --- | --- |
| Warm-up | Decoder + 1×1-Segmentierungs-Head | neuer Teil lernt, die ResNet-Features zu lesen |
| Fine-Tuning | zusätzlich `layer4` | letzte, abstrakte Backbone-Merkmale passen sich an Personenmasken an |

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.layers(x)


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.block = ConvBlock(in_channels + skip_channels, out_channels)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        return self.block(torch.cat([x, skip], dim=1))


class ResNet18UNet(nn.Module):
    def __init__(self, use_pretrained=True):
        super().__init__()
        weights = ResNet18_Weights.DEFAULT if use_pretrained else None
        encoder = resnet18(weights=weights)
        self.stem = nn.Sequential(encoder.conv1, encoder.bn1, encoder.relu)
        self.pool = encoder.maxpool
        self.layer1, self.layer2 = encoder.layer1, encoder.layer2
        self.layer3, self.layer4 = encoder.layer3, encoder.layer4

        self.decode3 = DecoderBlock(512, 256, 256)
        self.decode2 = DecoderBlock(256, 128, 128)
        self.decode1 = DecoderBlock(128, 64, 64)
        self.decode0 = DecoderBlock(64, 64, 64)
        self.final_block = ConvBlock(64, 32)
        self.segmentation_head = nn.Conv2d(32, 2, kernel_size=1)

        self.backbone_frozen = True
        for module in [self.stem, self.layer1, self.layer2, self.layer3, self.layer4]:
            for parameter in module.parameters():
                parameter.requires_grad = False
        self.register_buffer('image_mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('image_std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x):
        input_size = x.shape[-2:]
        x = (x - self.image_mean) / self.image_std
        x0 = self.stem(x)                 # 1/2 der Bildgröße
        x1 = self.layer1(self.pool(x0))   # 1/4
        x2 = self.layer2(x1)              # 1/8
        x3 = self.layer3(x2)              # 1/16
        x4 = self.layer4(x3)              # 1/32
        x = self.decode3(x4, x3)
        x = self.decode2(x, x2)
        x = self.decode1(x, x1)
        x = self.decode0(x, x0)
        x = F.interpolate(x, size=input_size, mode='bilinear', align_corners=False)
        return self.segmentation_head(self.final_block(x))

    def unfreeze_last_block(self):
        self.backbone_frozen = False
        for parameter in self.layer4.parameters():
            parameter.requires_grad = True

    def train(self, mode=True):
        super().train(mode)
        if self.backbone_frozen:
            for module in [self.stem, self.layer1, self.layer2, self.layer3, self.layer4]:
                module.eval()  # eingefrorene BatchNorm-Statistiken bewahren
        else:
            for module in [self.stem, self.layer1, self.layer2, self.layer3]:
                module.eval()
        return self


model = ResNet18UNet(USE_PRETRAINED_BACKBONE).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'Trainierbare Parameter im Warm-up: {trainable:,}')

## Loss: Cross Entropy und Dice Loss

Der Hintergrund nimmt viel mehr Pixel ein als die Personen. Ein Loss nur über die Pixelanzahl könnte deshalb eine fast leere Maske zu attraktiv finden. Wir kombinieren zwei Perspektiven:

1. **Cross Entropy (CE)** behandelt jedes Pixel als kleine Zwei-Klassen-Klassifikation. Für die richtige Klasse soll die Softmax-Wahrscheinlichkeit groß werden: $L_{CE}=-\frac{1}{N}\sum_{i=1}^{N}\log p_{i,y_i}$. `CrossEntropyLoss` erwartet rohe Logits, keine vorher berechnete Softmax.
2. **Dice Loss** vergleicht die vorhergesagte Personenfläche direkt mit der echten Personenfläche. Mit weichen Personenwahrscheinlichkeiten $p_i$ und Binärlabel $y_i$ ist $Dice=\frac{2\sum_i p_i y_i + \varepsilon}{\sum_i p_i + \sum_i y_i + \varepsilon}$ und $L_{Dice}=1-Dice$. Er belohnt Überlappung und hilft bei ungleich großen Klassen.

Wir optimieren $L=L_{CE}+L_{Dice}$. Für die Auswertung verwenden wir zusätzlich **IoU** (Intersection over Union): $IoU=\frac{|Vorhersage \cap Label|}{|Vorhersage \cup Label|}$. Anders als der Loss wird IoU nach einer harten Klassenentscheidung berechnet und ist daher gut interpretierbar.

In [ ]:
def dice_loss(logits, masks, smooth=1.0):
    person_probability = torch.softmax(logits, dim=1)[:, 1]
    target = masks.float()
    intersection = (person_probability * target).sum(dim=(1, 2))
    denominator = person_probability.sum(dim=(1, 2)) + target.sum(dim=(1, 2))
    dice = (2 * intersection + smooth) / (denominator + smooth)
    return 1 - dice.mean()


cross_entropy = nn.CrossEntropyLoss(weight=torch.tensor([0.35, 1.0], device=device))

def segmentation_loss(logits, masks):
    ce = cross_entropy(logits, masks)
    dice = dice_loss(logits, masks)
    return ce + dice, ce.detach(), dice.detach()


def foreground_iou(prediction, target):
    prediction, target = prediction.bool(), target.bool()
    intersection = (prediction & target).sum(dim=(1, 2)).float()
    union = (prediction | target).sum(dim=(1, 2)).float()
    return ((intersection + 1e-6) / (union + 1e-6)).mean().item()


def run_epoch(model, loader, optimizer=None):
    is_training = optimizer is not None
    model.train(is_training)
    totals = {'loss': 0.0, 'ce': 0.0, 'dice': 0.0, 'iou': 0.0, 'images': 0}
    context = torch.enable_grad() if is_training else torch.no_grad()
    with context:
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)
            if is_training:
                optimizer.zero_grad()
            logits = model(images)
            loss, ce, dice = segmentation_loss(logits, masks)
            if is_training:
                loss.backward()
                optimizer.step()
            batch_size = len(images)
            totals['loss'] += loss.item() * batch_size
            totals['ce'] += ce.item() * batch_size
            totals['dice'] += dice.item() * batch_size
            totals['iou'] += foreground_iou(logits.argmax(dim=1), masks) * batch_size
            totals['images'] += batch_size
    return {key: value / totals['images'] for key, value in totals.items() if key != 'images'}


def train_for_epochs(model, epochs, optimizer, history, phase):
    for epoch in range(epochs):
        train_values = run_epoch(model, train_loader, optimizer)
        valid_values = run_epoch(model, valid_loader)
        history.append({'phase': phase, 'epoch': len(history) + 1,
                        **{f'train_{k}': v for k, v in train_values.items()},
                        **{f'valid_{k}': v for k, v in valid_values.items()}})
        print(f"{phase}, Epoche {epoch + 1:02d}: "
              f"Train-Loss={train_values['loss']:.3f}, Valid-Loss={valid_values['loss']:.3f}, "
              f"Valid-IoU={valid_values['iou']:.3f}")
    return history

## 1. Warm-up: neuen Decoder trainieren

Der ResNet-18-Encoder bleibt zunächst eingefroren. Dadurch muss der neue Decoder nicht gleichzeitig mit sich verändernden Bildmerkmalen umgehen. Beim ersten Durchlauf lädt `torchvision` die ImageNet-Gewichte einmalig in den lokalen Cache. Falls das nicht möglich ist, setze `USE_PRETRAINED_BACKBONE = False`; dann startet der Encoder zufällig und braucht mehr Training.

In [ ]:
history = []
head_optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad), lr=1e-3, weight_decay=1e-4
)
history = train_for_epochs(model, HEAD_EPOCHS, head_optimizer, history, 'Decoder-Warm-up')

## 2. Fine-Tuning: letzten Backbone-Block mitlernen

Jetzt öffnen wir ausschließlich `layer4`, den letzten und abstraktesten ResNet-Block. Die Lernrate sinkt deutlich. Frühe Kanten- und Farbmerkmale bleiben unverändert, während die tiefen Merkmale etwas spezifischer für Personen in diesen Szenen werden dürfen.

In [ ]:
model.unfreeze_last_block()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainierbare Parameter nach Freigabe von layer4: {trainable:,}')

fine_tune_optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad), lr=1e-4, weight_decay=1e-4
)
history = train_for_epochs(model, FINE_TUNE_EPOCHS, fine_tune_optimizer, history, 'Fine-Tuning layer4')

In [ ]:
epochs = [row['epoch'] for row in history]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, [row['train_loss'] for row in history], marker='o', label='Training')
axes[0].plot(epochs, [row['valid_loss'] for row in history], marker='o', label='Validierung')
axes[0].set(xlabel='Epoche', ylabel='CE + Dice Loss', title='Loss-Verlauf')
axes[0].legend()
axes[1].plot(epochs, [row['train_iou'] for row in history], marker='o', label='Training')
axes[1].plot(epochs, [row['valid_iou'] for row in history], marker='o', label='Validierung')
axes[1].set(xlabel='Epoche', ylabel='Foreground-IoU', ylim=(0, 1), title='Maskenüberlappung')
axes[1].legend()
for axis in axes:
    axis.axvline(HEAD_EPOCHS + 0.5, color='gray', linestyle='--', alpha=0.7)
plt.tight_layout()

## Vorhersagen verstehen

Gelb ist die echte Maske, Cyan die Vorhersage. Die Überlagerung zeigt sofort, ob eine Person vollständig erkannt wurde oder ob das Modell Hintergrund und Person verwechselt. Ein guter Loss allein genügt nicht: Gerade bei Segmentierung sollte man mehrere Masken auch visuell prüfen.

In [ ]:
model.eval()
images, masks = next(iter(valid_loader))
with torch.no_grad():
    predictions = model(images.to(device)).argmax(dim=1).cpu()

fig, axes = plt.subplots(3, 4, figsize=(12, 10))
for row, (image, target, prediction) in enumerate(zip(images[:4], masks[:4], predictions[:4])):
    axes[0, row].imshow(image.permute(1, 2, 0))
    axes[0, row].set_title(f'Foto {row + 1}')
    axes[1, row].imshow(target, cmap='magma', vmin=0, vmax=1)
    axes[1, row].set_title('Label')
    axes[2, row].imshow(image.permute(1, 2, 0))
    axes[2, row].imshow(prediction, cmap='cool', vmin=0, vmax=1, alpha=0.45)
    axes[2, row].set_title(f'Vorhersage, IoU={foreground_iou(prediction.unsqueeze(0), target.unsqueeze(0)):.2f}')
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()

## Zusammenfassung

- Segmentierung ordnet **jedem Pixel** eine Klasse zu; hier Hintergrund oder Person.
- Penn-Fudan liefert echte Instanzmasken. Für die semantische Aufgabe machen wir daraus eine binäre Personenmaske.
- Der vortrainierte ResNet-18-Encoder liefert allgemeine Bildmerkmale; der neue Decoder stellt daraus wieder eine dichte Pixelkarte her.
- Cross Entropy belohnt richtige Pixelklassen, Dice Loss die Flächenüberlappung. IoU macht die Qualität der fertigen Masken messbar.
- Beim Fine-Tuning lernt nur der letzte Backbone-Block mit kleiner Lernrate weiter.